# Gun 1 - Embedding'lerin FAISS'e Kaydedilmesi

In [1]:
import sys
import os
import json

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from embedder import embed_chunks
from vector_store import build_index, save_index, load_index, search, load_index_path

print("vector_store yuklendi.")

vector_store yuklendi.


## 1. Gun 2'de uretilen embedding'leri yukle ve index kur

In [2]:
with open("../data/processed/chunk_embeddings.json", encoding="utf-8") as f:
    embedded_chunks = json.load(f)

index, metadata = build_index(embedded_chunks)
print(f"{index.ntotal} vektor index'e eklendi, boyut: {index.d}")

11 vektor index'e eklendi, boyut: 384


## 2. Diske kaydet ve geri yukle

config/settings.yaml icindeki vector_db.path kullaniliyor.

In [3]:
INDEX_PATH = os.path.join("..", load_index_path())
save_index(index, metadata, INDEX_PATH)
print(f"Index kaydedildi: {INDEX_PATH}.faiss ve {INDEX_PATH}.meta.json")

loaded_index, loaded_metadata = load_index(INDEX_PATH)
assert loaded_index.ntotal == index.ntotal
assert loaded_metadata == metadata
print("OK - geri yuklenen index, orijinaliyle birebir ayni")

Index kaydedildi: ..\./data/processed/faiss_index.faiss ve ..\./data/processed/faiss_index.meta.json
OK - geri yuklenen index, orijinaliyle birebir ayni


## 3. Anlamsal arama testi

Bir soruyu embedding'e cevirip index'te en yakin chunk'lari buluyoruz.

In [4]:
query = "Kimin monitor talebi var?"
query_embedding = embed_chunks([{"chunk_id": -1, "text": query, "token_count": 0}])[0]["embedding"]

results = search(loaded_index, loaded_metadata, query_embedding, top_k=3)
for r in results:
    print(f"skor {r['score']:.4f} | chunk {r['chunk_id']}: {r['text'][:80]}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

skor 0.4644 | chunk 10: Pazarlama
Konu: Ek Ekran Talebi

Tasarim islerinde verimliligi artirmak icin iki
skor 0.4293 | chunk 3: tahsis edilmesini rica ederim.

---

Talep Eden: Ahmet Yilmaz
Tarih: 2026-08-13

skor 0.4199 | chunk 5: yeni bir laptop tahsis edilmesini rica ederim.

---

Talep Eden: Zeynep Kaya
Tar


In [5]:
# En yuksek skorlu sonuc, ekran/monitor konulu (goruntu donanimi) bir chunk olmali.
# Not: en alakali sonuc literal "monitor" kelimesini iceren chunk (id 0) degil,
# "Ek Ekran Talebi" chunk'i (id 10) cikiyor. Bu, embedding'in kelime eslesmesi degil
# KONU eslesmesi yaptigini gosteriyor: chunk 0 monitor kelimesini icerse de, cevresinde
# konuyla alakasiz uzun bir cumle (yapay zeka modulu test sureci) oldugu icin vektoru
# biraz "sulanmis"; chunk 10 ise net ve kisa bir ekran talebi oldugu icin sorguya daha yakin.
assert results[0]["chunk_id"] in (0, 10)
print("OK - en alakali sonuc, ekran/monitor konulu bir chunk (konu bazli eslesme)")

OK - en alakali sonuc, ekran/monitor konulu bir chunk (konu bazli eslesme)
